# **Creating Personal Assistant:**
**RAG system with LLM to analyse judicial court document**

1 - Installing required packages

In [50]:
!pip install -q -U watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.8 MB/s eta 0:00:00


In [1]:
!pip install -q -r drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap5/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 599.1/599.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.1/420.1 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [15]:
!ls drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap5/

Archive  BKP  LLM_GPT_PERSONAL_ASSISTANT_WITH_RAG.ipynb  requirements.txt


In [2]:
# Import packages
import os
import openai
import langchain
import chromadb
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.llms import OpenAI
from langchain.chains.question_answering import load_qa_chain
import warnings
warnings.filterwarnings('ignore')

In [54]:
%reload_ext watermark
%watermark -a "João Machado"

Author: João Machado



2 - Extracting arquive's text

In [3]:
# Function to read the pdf's text
def read_pdf_text(file_path):

  #Access the folder with the pdfs
  loader = PyPDFDirectoryLoader(file_path)

  # Reads the pdfs
  documents = loader.load()

  # Returns the text
  return documents


In [12]:
# Execute the function
court_doc = read_pdf_text('drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap5/Archive/')

In [13]:
# Print content
court_doc

[Document(metadata={'producer': 'Microsoft® Word 2010; modified using iText® 5.5.12 ©2000-2017 iText Group NV (AGPL-version)', 'creator': 'Microsoft® Word 2010', 'creationdate': '2024-03-01T15:20:11+00:00', 'title': 'Concl', 'author': 'Margarida Reis', 'moddate': '2024-03-01T15:23:18+00:00', 'source': 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap5/Archive/court_doc.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1'}, page_content='Tribunal Adm inistrativo e Fiscal de Penafiel  \nU n i d ad e  Or g â n i c a  1 \nTRIBUNAL ADMINISTRATIVO E FISCAL DE PENAFIEL \nPraça do Município 28, 4560-481 Penafiel \nTelefone: 255718060 | Fax: 213506002 | Email: penafiel.taf@tribunais.org.pt \n 1 \n* \nProcesso n.º 1289/13.0BEPRT \n* \n* \nI. RELATÓRIO  \nJOSÉ CARLOS DA COSTA TOGA MACHADO , residente na Rua das Póvoas, n.º 625, 4440 -\n077 Campo – Valongo, representado em juízo pela sua tutora Maria do Carmo Martins Rocha \nToga, intentou, no Tribunal Administrativo  e Fiscal do Porto, a p

In [14]:
# Print content's lenght
len(court_doc)

38

3 - Creating the Embedding's generator using GPT API

In [19]:
# Check if .env file is present
dotenv_path = 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap5/.env'

if os.path.exists(dotenv_path):
    print(f"The .env file exists at: {dotenv_path}")
else:
    print(f"Error: The .env file does NOT exist at: {dotenv_path}")
    print("Please ensure the path is correct and the file is present.")

The .env file exists at: drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap5/.env


In [20]:
# Check if API KEY is present
from dotenv import load_dotenv

dotenv_path = 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap5/.env'
load_success = load_dotenv(dotenv_path)

print(f"load_dotenv() returned: {load_success}")

if 'OPENAI_API_KEY' in os.environ:
    print('OPENAI_API_KEY is now available in the environment.')
else:
    print('OPENAI_API_KEY is NOT available in the environment. Please check your .env file content.')

load_dotenv() returned: True
OPENAI_API_KEY is now available in the environment.


In [21]:
# Creating the embeddings generator
embeddings_generator = OpenAIEmbeddings(api_key = os.environ['OPENAI_API_KEY'])
type(embeddings_generator)

langchain_community.embeddings.openai.OpenAIEmbeddings

In [25]:
# Testing embeddings generator
embeddings_generator.embed_query("Qual foi a conclusão do documento do Tribunal Administrativo e Fiscal de Penafiel relativo ao José Carlos no processo do hospital S. João?")

[0.013021740797518486,
 0.026458931548605936,
 0.006017524160972694,
 -0.013333327797033933,
 -0.01626743901957857,
 0.003687112982819873,
 -0.0003071241891401613,
 -0.0015043810402431735,
 -0.010658872096977981,
 0.005806553990742976,
 0.007140535716003161,
 0.0008592985512050214,
 0.004758193526484858,
 0.003547548011425352,
 -0.015345660657458281,
 -0.002018824178637375,
 0.027471588365708583,
 -0.013956501796064825,
 0.0056475148318787406,
 -0.016903595655035512,
 -0.009853939325337265,
 -0.006004541524546643,
 -0.03487177867287787,
 -0.0010629661630680645,
 0.01307367134322269,
 0.005985067104246289,
 -0.0023839652519021984,
 -0.0242258901436485,
 0.0042810755842308725,
 -0.0009712751291558848,
 0.014203174682127461,
 -0.004268092947804822,
 -0.01473546929485344,
 -0.003044464796319265,
 -0.011463805799941254,
 0.005576109400212905,
 0.0055014583094405555,
 0.006913336784579602,
 0.02342095737200778,
 0.004767930503804397,
 0.014969159544490025,
 0.0007818075087144739,
 0.00030732

4 - Creating the Vector Store for the Embeddings context

In [26]:
# Defining index
index_name = 'court-doc-index'

In [29]:
# Creating vector store
# Generate unique IDs for each document
document_ids = [f"doc_{i}" for i in range(len(court_doc))]

court_vector_store = Chroma.from_documents(
    documents=court_doc,
    embedding=embeddings_generator,
    ids=document_ids,
    collection_name=index_name
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [31]:
type(court_vector_store)

langchain_community.vectorstores.chroma.Chroma

In [32]:
# Defining a function to retrive the vectors by similarity
def court_similarity_search(query, k=3):

    # Matching results
    matching_results = court_vector_store.similarity_search(
        query,
        k=k
    )

    return matching_results

5 - Creating Personal Assistant with LangChain

In [43]:
# Creating llm instance
llm_instance = OpenAI(openai_api_key=os.environ['OPENAI_API_KEY'], temperature = 0)

In [44]:
# Creating a chain
chain = load_qa_chain(llm_instance, chain_type='stuff')

In [45]:
# Defining a function to get a response
def getting_response(query):

    # Matching results
    matching_results = court_similarity_search(query)

    # Executing chain and getting response
    response = chain.run(input_documents=matching_results, question=query)

    return response

6 - Executing AI assistant and chatting with PDF


In [47]:
# Defining prompt1
prompt1 = 'Qual foi a conclusão do documento do Tribunal Administrativo e Fiscal de Penafiel relativo ao José Carlos no processo do hospital S. João?'
prompt2 = 'Qual o valor a pagar a José Carlos pelo tribunal?'
prompt3 = 'Baseado no documento do tribunal, e em situações semelhantes, qual a probabilidade de José Carlos ser recompensado pelo o que se passou?'

In [48]:
# Getting the answer from the prompt
response1 = getting_response(prompt1)
response2 = getting_response(prompt2)
response3 = getting_response(prompt3)
print(response1)
print(response2)
print(response3)

 A conclusão do documento foi que o autor, José Carlos, sofreu uma paragem cardiorrespiratória devido à negligência dos serviços do hospital S. João, o que resultou em uma encefalopatia anóxica e um défice funcional permanente. O autor está buscando uma condenação do réu no pagamento de uma quantia global de € 1.219.960,22, acrescida de juros legais.
 O valor a pagar a José Carlos pelo tribunal é de € 200.000,00, atualizado à presente data, a título de indemnização pelos danos morais sofridos. Além disso, o tribunal também irá condenar o réu a pagar uma indemnização ilíquida, com o limite máximo de € 789.870,25, para cobrir as despesas médicas e de cuidados futuros do autor.
 Não é possível determinar a probabilidade de José Carlos ser recompensado com base apenas no documento do tribunal e em situações semelhantes. A decisão final dependerá de vários fatores, incluindo a avaliação do tribunal sobre a responsabilidade do Réu e a extensão dos danos sofridos pelo Autor.


In [55]:
%watermark -v -m

Python implementation: CPython
Python version       : 3.12.12
IPython version      : 7.34.0

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.6.113+
Machine     : x86_64
Processor   : x86_64
CPU cores   : 2
Architecture: 64bit



In [56]:
%watermark -a "João Machado"

Author: João Machado

